# 🚀 [ICML 2026] LiDAR: Dual-GPU Parallel Accelerated on Kaggle
### Tái lập Thực nghiệm: `LiDAR (DPM-5 / n=50)` trên GenEval Benchmark (Chạy song song 2x GPU T4)

**Bài báo:** *Lookahead Sample Reward Guidance for Test-Time Scaling of Diffusion Models* ([arXiv:2602.03211](https://arxiv.org/abs/2602.03211))  
**GitHub Repository:** [github.com/leekwanreal/Noisy-Reward](https://github.com/leekwanreal/Noisy-Reward)  

**⚡ Ưu điểm trên Kaggle:**
1. **Tận dụng 2x GPU T4:** Chia đôi prompt chẵn/lẻ $\implies$ Chạy xong toàn bộ 553 prompts cả 2 Pha chỉ trong **~3.8 tiếng**.
2. **Cơ chế Tự Động Khôi Phục Từ File ZIP / Output Cũ:** Tự động phát hiện và giải nén file `.zip` đã xuất từ trước để chạy nối tiếp ngay lập tức mà không mất công làm lại.

## 📦 Step 1: Kiểm tra 2x GPU & Cài đặt Môi trường Chuẩn (Có Tự Động Giải Nén ZIP Cũ)

In [ ]:
# 1. Kiểm tra 2 GPU
!nvidia-smi

# 2. Tải mã nguồn Noisy-Reward về Kaggle
import os, shutil, glob
%cd /kaggle/working
if not os.path.exists("/kaggle/working/Noisy-Reward"):
    !git clone https://github.com/leekwanreal/Noisy-Reward.git
%cd /kaggle/working/Noisy-Reward
!git pull origin main

# 3. Tự động đồng bộ toàn bộ dữ liệu từ /kaggle/input sang /kaggle/working
os.makedirs("/kaggle/working/LiDAR_Experiment/Lookahead_samples", exist_ok=True)
os.makedirs("/kaggle/working/LiDAR_Experiment/Target_samples", exist_ok=True)

# 3a. Quét và giải nén file .zip phiên cũ nếu có
zip_files = glob.glob("/kaggle/input/**/*.zip", recursive=True)
for zf in zip_files:
    print(f"📦 Tìm thấy file zip dữ liệu cũ: {zf}. Đang giải nén để chạy nối tiếp...")
    try:
        shutil.unpack_archive(zf, "/kaggle/working")
    except Exception as e:
        print(f"⚠️ Lỗi giải nén: {e}")

# 3b. Tự động sao chép tất cả các thư mục Lookahead từ input
for l_dir in glob.glob("/kaggle/input/**/Lookahead_samples/*", recursive=True):
    if os.path.isdir(l_dir):
        base_name = os.path.basename(l_dir)
        dest_dir = os.path.join("/kaggle/working/LiDAR_Experiment/Lookahead_samples", base_name)
        os.makedirs(dest_dir, exist_ok=True)
        for p_dir in glob.glob(os.path.join(l_dir, "[0-9]*")):
            p_name = os.path.basename(p_dir)
            p_dest = os.path.join(dest_dir, p_name)
            if not os.path.exists(p_dest) and os.path.isdir(p_dir):
                shutil.copytree(p_dir, p_dest)

# 3c. Tự động sao chép tất cả các thư mục Target từ input
for t_dir in glob.glob("/kaggle/input/**/Target_samples/*", recursive=True):
    if os.path.isdir(t_dir):
        base_name = os.path.basename(t_dir)
        parts = base_name.split("_")
        canonical_name = "_".join(parts[:8]) if len(parts) >= 9 else base_name
        dest_dir = os.path.join("/kaggle/working/LiDAR_Experiment/Target_samples", canonical_name)
        os.makedirs(dest_dir, exist_ok=True)
        for p_dir in glob.glob(os.path.join(t_dir, "[0-9]*")):
            p_name = os.path.basename(p_dir)
            p_dest = os.path.join(dest_dir, p_name)
            if not os.path.exists(p_dest) and os.path.isdir(p_dir):
                shutil.copytree(p_dir, p_dest)

n_look = len(glob.glob("/kaggle/working/LiDAR_Experiment/Lookahead_samples/*/[0-9]*"))
n_targ = len(glob.glob("/kaggle/working/LiDAR_Experiment/Target_samples/*/[0-9]*"))
print(f"\n✅ Đã đồng bộ xong dữ liệu: {n_look} Lookahead prompts, {n_targ} Target prompts sẵn sàng!")

# 4. Cài đặt các thư viện cần thiết
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas

# 5. Tải file vocab cho hpsv2
import urllib.request, hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)

print("\n✅ Môi trường trên Kaggle đã được cài đặt hoàn tất!")

## ⚡ Step 2: Phase 1 — Lookahead Sampling (Chạy Song Song trên cả 2 GPU T4)
- Tự động chia đôi 553 prompts chạy song song trên GPU 0 và GPU 1.
- Tự động kiểm tra `is_lookahead_complete()` để bỏ qua các prompt đã làm từ trước trong $0.001\text{s}$.

In [ ]:
%cd /kaggle/working/Noisy-Reward
import torch
n_gpus = torch.cuda.device_count()
print(f"🚀 Tìm thấy {n_gpus} GPU khả dụng!")

if n_gpus >= 2:
    print("⚡ Đang khởi động 2 tiến trình chạy song song trên GPU 0 và GPU 1 (hiển thị log trực tiếp)...")
    !CUDA_VISIBLE_DEVICES=0 python lookahead_sampling.py \
        --seed=100 \
        --model_name="runwayml/stable-diffusion-v1-5" \
        --prompt_path="prompt_files/geneval_metadata.jsonl" \
        --output_dir="/kaggle/working/LiDAR_Experiment/Lookahead_samples" \
        --num_particles=50 \
        --num_inference_steps=5 \
        --metrics_to_compute="ImageReward" \
        --num_splits=2 \
        --split_idx=0 & \
    CUDA_VISIBLE_DEVICES=1 python lookahead_sampling.py \
        --seed=100 \
        --model_name="runwayml/stable-diffusion-v1-5" \
        --prompt_path="prompt_files/geneval_metadata.jsonl" \
        --output_dir="/kaggle/working/LiDAR_Experiment/Lookahead_samples" \
        --num_particles=50 \
        --num_inference_steps=5 \
        --metrics_to_compute="ImageReward" \
        --num_splits=2 \
        --split_idx=1 & \
    wait
else:
    print("⚡ Đang chạy trên 1 GPU...")
    !python lookahead_sampling.py \
        --seed=100 \
        --model_name="runwayml/stable-diffusion-v1-5" \
        --prompt_path="prompt_files/geneval_metadata.jsonl" \
        --output_dir="/kaggle/working/LiDAR_Experiment/Lookahead_samples" \
        --num_particles=50 \
        --num_inference_steps=5 \
        --metrics_to_compute="ImageReward"

print("\n✅ Đã hoàn thành 100% Pha 1 trên toàn bộ 553 prompts!")

## 🎯 Step 3: Phase 2 — LiDAR Target Sampling (Chạy Song Song trên cả 2 GPU T4)
- Sử dụng 50 hạt Lookahead đã sinh ở Pha 1 làm ngân hàng dẫn đường.
- Tự động kiểm tra `is_target_complete()` để bỏ qua các prompt đã làm từ trước trong $0.001\text{s}$.

In [ ]:
%cd /kaggle/working/Noisy-Reward
import torch
n_gpus = torch.cuda.device_count()

if n_gpus >= 2:
    print("⚡ Đang khởi động 2 tiến trình sinh ảnh song song trên GPU 0 và GPU 1 (hiển thị log trực tiếp)...")
    !CUDA_VISIBLE_DEVICES=0 python LiDAR_sampling.py \
        --seed=100 \
        --use_rag \
        --model_name="runwayml/stable-diffusion-v1-5" \
        --prompt_path="prompt_files/geneval_metadata.jsonl" \
        --output_dir="/kaggle/working/LiDAR_Experiment/Target_samples" \
        --num_inference_steps=50 \
        --num_particles=4 \
        --top_k=50 \
        --scale=12.5 \
        --resample_t_end=200 \
        --lookahead_path="/kaggle/working/LiDAR_Experiment/Lookahead_samples/100_50_5" \
        --metrics_to_compute="ImageReward" \
        --save_individual_images \
        --num_splits=2 \
        --split_idx=0 & \
    CUDA_VISIBLE_DEVICES=1 python LiDAR_sampling.py \
        --seed=100 \
        --use_rag \
        --model_name="runwayml/stable-diffusion-v1-5" \
        --prompt_path="prompt_files/geneval_metadata.jsonl" \
        --output_dir="/kaggle/working/LiDAR_Experiment/Target_samples" \
        --num_inference_steps=50 \
        --num_particles=4 \
        --top_k=50 \
        --scale=12.5 \
        --resample_t_end=200 \
        --lookahead_path="/kaggle/working/LiDAR_Experiment/Lookahead_samples/100_50_5" \
        --metrics_to_compute="ImageReward" \
        --save_individual_images \
        --num_splits=2 \
        --split_idx=1 & \
    wait
else:
    print("⚡ Đang chạy trên 1 GPU...")
    !python LiDAR_sampling.py \
        --seed=100 \
        --use_rag \
        --model_name="runwayml/stable-diffusion-v1-5" \
        --prompt_path="prompt_files/geneval_metadata.jsonl" \
        --output_dir="/kaggle/working/LiDAR_Experiment/Target_samples" \
        --num_inference_steps=50 \
        --num_particles=4 \
        --top_k=50 \
        --scale=12.5 \
        --resample_t_end=200 \
        --lookahead_path="/kaggle/working/LiDAR_Experiment/Lookahead_samples/100_50_5" \
        --metrics_to_compute="ImageReward" \
        --save_individual_images

print("\n✅ Đã hoàn thành 100% Pha 2 sinh ảnh đích trên toàn bộ 553 prompts!")

## 📊 Step 4: Đánh giá Toàn diện (ImageReward, CLIP, HPS v2.1) & Đối chứng Bảng 2

In [ ]:
import os, glob, json, gc, torch, sys
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# 1. Tìm thư mục kết quả mới nhất
target_runs = sorted(glob.glob("/kaggle/working/LiDAR_Experiment/Target_samples/*"))
if not target_runs:
    # Fallback tìm các thư mục khác
    target_runs = sorted(glob.glob("/content/LiDAR_Experiment/Target_samples/*") + glob.glob("Target_samples/*"))
if not target_runs:
    raise FileNotFoundError("Chưa tìm thấy thư mục kết quả. Hãy đảm bảo Step 3 đã chạy xong!")

latest_dir = target_runs[-1]
print(f"📂 Đang phân tích kết quả tại: {latest_dir}")

# 2. Thu thập danh sách ảnh và điểm ImageReward
all_images = []
all_prompts = []
ir_mean_scores = []
ir_best_scores = []
prompt_dirs = sorted([d for d in glob.glob(os.path.join(latest_dir, "*")) if os.path.isdir(d) and os.path.basename(d).isdigit()])

for p_dir in prompt_dirs:
    meta_path = os.path.join(p_dir, "metadata.jsonl")
    results_path = os.path.join(p_dir, "results.json")
    prompt_text = ""
    if os.path.exists(meta_path):
        try:
            with open(meta_path, "r", encoding="utf-8") as f:
                prompt_text = json.load(f).get("prompt", "")
        except Exception:
            pass
    if os.path.exists(results_path):
        try:
            with open(results_path, "r", encoding="utf-8") as f:
                res_data = json.load(f)
                ir_mean_scores.append(res_data.get("ImageReward", {}).get("mean", 0.0))
                ir_best_scores.append(res_data.get("ImageReward", {}).get("max", res_data.get("ImageReward", {}).get("mean", 0.0)))
        except Exception:
            pass

    # Tìm chính xác các ảnh thành phẩm trong thư mục samples/
    sample_imgs = sorted(glob.glob(os.path.join(p_dir, "samples", "*.png")))
    if not sample_imgs:
        sample_imgs = sorted([p for p in glob.glob(os.path.join(p_dir, "*.png")) if "grid" not in p])

    for img_path in sample_imgs:
        all_images.append(img_path)
        all_prompts.append(prompt_text)

ir_mean = sum(ir_mean_scores) / max(1, len(ir_mean_scores))
ir_best_mean = sum(ir_best_scores) / max(1, len(ir_best_scores))
print(f"🖼️ Tổng số ảnh sinh ra: {len(all_images)} ảnh trên {len(prompt_dirs)} prompts.")

# 3. Tính CLIP Score tuần tự
if len(all_images) > 0:
    print("\n⏳ Đang tính CLIP-Score...")
    try:
        from fks_utils import do_clip_score
    except ImportError:
        sys.path.insert(0, '/kaggle/working/Noisy-Reward/fkd_diffusers')
        from fks_utils import do_clip_score

    clip_scores = []
    for idx in tqdm(range(0, len(all_images), 10)):
        batch_imgs = [Image.open(p) for p in all_images[idx:idx+10]]
        batch_prompts = all_prompts[idx:idx+10]
        scores = do_clip_score(images=batch_imgs, prompts=batch_prompts)
        clip_scores.extend(scores)
    clip_mean = sum(clip_scores) / max(1, len(clip_scores))
else:
    clip_mean = 0.0

# 4. Tính HPS v2.1 Score tuần tự
if len(all_images) > 0:
    print("\n⏳ Đang tính HPS v2.1 Score...")
    from fkd_diffusers.rewards import do_human_preference_score
    hps_scores = []
    for idx in tqdm(range(0, len(all_images), 10)):
        batch_imgs = [Image.open(p) for p in all_images[idx:idx+10]]
        batch_prompts = all_prompts[idx:idx+10]
        scores = do_human_preference_score(images=batch_imgs, prompts=batch_prompts)
        hps_scores.extend(scores)
    hps_mean = sum(hps_scores) / max(1, len(hps_scores))
else:
    hps_mean = 0.0

# Giải phóng bộ nhớ GPU
gc.collect()
torch.cuda.empty_cache()

# 5. In bảng đối chứng toàn diện so với Bảng 2 bài báo
print("\n" + "="*78)
print("📈 KẾT QUẢ ĐỐI CHỨNG THỰC NGHIỆM VS BÀI BÁO (TABLE 2 - SD v1.5 LiDAR DPM-5/n=50)")
print("="*78)
print(f"• ImageReward (All 4 Particles Mean):  {ir_mean:.4f}  | Bài báo Table 2: 0.378 ~ 0.384")
print(f"• ImageReward (Best-of-4 Rank 1):     {ir_best_mean:.4f}  | Bài báo Table 8 BoN+LiDAR: 0.969")
print(f"• CLIP Score:                         {clip_mean:.4f}  | Bài báo Table 2: 0.278")
print(f"• HPS v2.1:                           {hps_mean:.4f}  | Bài báo Table 2: 0.272 ~ 0.275")
print("="*78)

# 6. Hiển thị ảnh mẫu
sample_grid = os.path.join(latest_dir, "00000/grid.png")
if os.path.exists(sample_grid):
    plt.figure(figsize=(16, 5))
    plt.imshow(Image.open(sample_grid))
    plt.axis("off")
    plt.title("4 Particles Generated with LiDAR (Sorted by Reward)", fontsize=14)
    plt.show()

## 💾 Step 5: Nén & Tải Toàn bộ Kết quả về Máy tính
Nén toàn bộ thư mục thực nghiệm (Lookahead + Target) thành file `.zip` để tải về máy từ mục **Output (bên phải)**.

In [ ]:
!zip -r -q /kaggle/working/LiDAR_Full_Experiment.zip /kaggle/working/LiDAR_Experiment
print("\n✅ Đã nén xong toàn bộ kết quả thành công: /kaggle/working/LiDAR_Full_Experiment.zip")
print("📥 Bạn có thể tải file ZIP về máy tính tại mục 'Output' ở cột bên phải giao diện Kaggle!")

## 🧪 (Tùy chọn) Chạy Bộ 3 Bài Test Lipschitz & Phân tích Đột phá
Chạy đo đạc độc lập 3 bài test lý thuyết để vẽ biểu đồ so sánh giữa LiDAR gốc vs Phương pháp của bạn.

In [ ]:
%cd /kaggle/working/Noisy-Reward

!python test_lidar_vs_smoothed_surrogate.py \
    --num_prompts=-1 \
    --num_particles=20 \
    --sigma=0.05 \
    --output_dir="/kaggle/working/test_results"